# Proyecto BATSÚ — Notebook 03b: Redacción del corpus con el modelo generador (vLLM)

**Universidad Cenfotec — MISIA** · Alvaro Granados, Jose Leonel Hernández

El notebook 02 produjo las transiciones estructuradas: para cada episodio se sabe qué acción tomó el agente en cada turno, cómo reaccionó el turista (estado afectivo, etapa AIDA) y cómo terminó. Lo que falta es el **texto de los mensajes**, que es lo único que el Juez RLAIF puede leer.

Este notebook lo redacta con el modelo generador abierto fijado en la Tabla 5.2, **Mistral-7B-Instruct v0.3**, ejecutado en la GPU de Colab con el motor de inferencia **vLLM**. Esta variante sustituye a la generación con Transformers del notebook 03: vLLM procesa todos los episodios en paralelo con *continuous batching*, sin que cada lote espere al episodio más largo, lo que en el piloto redujo el costo de ~12 s a 1.96 s por episodio con lotes de 128 en una L4. El modelo no decide nada: recibe la traza estructurada del episodio ya resuelta por la tabla de reacción (decisión D3) y devuelve únicamente el diálogo en formato JSON. Es de familia distinta al juez (Llama 3) para evitar el sesgo de autopreferencia (decisión D4).

**Modo piloto.** La sección 2 tiene los parámetros `N_SEMILLAS` y `N_CORRIDAS`; por defecto redacta **una semilla, una corrida** para medir el tiempo por episodio y revisar la calidad del texto. La corrida completa se lanza cambiando esos dos valores a `None`. El guardado es incremental (una línea JSON por episodio), de modo que si la sesión de Colab se corta, al volver a correr solo se redactan los episodios que faltan.

**Secciones**
1. Configuración: Drive, GPU, librerías y acceso a Hugging Face
2. Parámetros de la corrida
3. Carga del corpus estructurado
4. Carga del modelo generador
5. Construcción del prompt a partir de la traza del episodio
6. Redacción por lotes con guardado incremental
7. Validación, muestra y manifiesto


## 1. Configuración

Se necesita una sesión de Colab **con GPU L4** (24 GB): el modelo se carga en 16 bits (14.5 GB de pesos) y el resto de la memoria se usa como caché de atención para procesar muchos episodios a la vez. En una T4 (15 GB) no cabe en 16 bits; ver la nota al final de esta sección.

Se instala `vllm`. La instalación tarda unos minutos y **al terminar hay que reiniciar la sesión** (Entorno de ejecución → Reiniciar sesión) para que la versión de PyTorch que vLLM trae quede activa; después se vuelve a correr desde la primera celda, y la instalación ya no se repite.

El token de Hugging Face se toma del secreto `HF_TOKEN` de Colab, igual que en el notebook 03.

> Si solo hay T4 disponible, cambiar `MODELO` en la Sección 2 por una versión AWQ de 4 bits del mismo modelo y añadir `quantization='awq'` en la carga; vLLM la ejecuta también con continuous batching.


In [26]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import sys, json, time, re
import numpy as np
import pandas as pd

NOMBRE_CARPETA = 'DatosBatsuv2'
candidatos = [p for p in Path('/content/drive/MyDrive').rglob(NOMBRE_CARPETA) if p.is_dir()]
assert candidatos, f'No se encontró la carpeta "{NOMBRE_CARPETA}"'
BASE_DIR = candidatos[0]
CORPUS_DIR = BASE_DIR / 'corpus'
sys.path.insert(0, str(BASE_DIR))

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), 'Este notebook requiere GPU: cambiar el tipo de entorno de ejecución'
print('GPU:', torch.cuda.get_device_name(0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NVIDIA L4, 23034 MiB
GPU: NVIDIA L4


In [27]:
import importlib
if importlib.util.find_spec('vllm') is None:
    print('Instalando vLLM (varios minutos)...')
    !pip install -q vllm 2>&1 | tail -1
    print('\n>>> Instalación terminada. REINICIAR la sesión (Entorno de ejecución → Reiniciar sesión) y volver a correr desde la primera celda.')
else:
    import vllm
    print('vLLM', vllm.__version__, 'ya instalado')

from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('Sesión de Hugging Face iniciada')


vLLM 0.31.0 ya instalado
Sesión de Hugging Face iniciada


In [28]:
!pip uninstall -y -q torchaudio
!pip install -q --no-deps torchvision --index-url https://download.pytorch.org/whl/cu130 2>&1 | tail -1
import torch, torchvision
print('torch', torch.__version__, '· torchvision', torchvision.__version__)

torch 2.13.0+cu130 · torchvision 0.28.0+cu130


## 2. Parámetros de la corrida

- **`ETIQUETA_CORPUS`**: qué corpus estructurado se redacta (el nombre que se le dio en el notebook 02).
- **`N_SEMILLAS`, `N_CORRIDAS`**: alcance del piloto. `1, 1` redacta un solo episodio; `None, None` redacta el corpus completo. En modo piloto se eligen episodios con al menos `MIN_TURNOS_PILOTO` turnos, para revisar un diálogo con varias interacciones y no uno que cierra en el primer mensaje.
- **`MODELO`**: identificador exacto en Hugging Face, que queda registrado en el manifiesto (decisión D4).
- **`IDIOMA_INTERNACIONAL`**: la Tabla 4.4 admite español neutro o inglés para el turista internacional. Se fija español neutro por defecto para que el juez evalúe todos los episodios en la misma lengua; cambiar a `'en'` si se quiere la variante bilingüe.
- **`TEMPERATURA`**: 0.7, el punto medio que la literatura de generación sintética recomienda entre repetición y deriva de formato.
- **`LOTE`**: episodios que se envían juntos a vLLM. Con vLLM el lote no limita el paralelismo (el motor decide cuántas secuencias procesa a la vez según la memoria); solo define cada cuántos episodios se guarda el archivo incremental.


In [36]:
ETIQUETA_CORPUS = 'v0_escala05_prior40'
N_SEMILLAS = None            # None = todas
N_CORRIDAS = None            # None = todas
MIN_TURNOS_PILOTO = 3     # en modo piloto se eligen episodios de al menos esta longitud, para ver un diálogo real
MODELO = 'mistralai/Mistral-7B-Instruct-v0.3'   # 16 bits; vLLM lo carga completo en la L4
IDIOMA_INTERNACIONAL = 'es'    # 'es' (español neutro) o 'en'
TEMPERATURA = 0.7
LOTE = 128                     # episodios que se envían juntos a vLLM (solo controla cada cuánto se guarda)
MAX_PALABRAS_MENSAJE = 40      # estilo WhatsApp (Sección 4.7)

RUTA_CORPUS = CORPUS_DIR / f'corpus_{ETIQUETA_CORPUS}.parquet'
RUTA_TEXTOS = CORPUS_DIR / f'textos_{ETIQUETA_CORPUS}.jsonl'      # una línea por episodio, guardado incremental
assert RUTA_CORPUS.exists(), f'No existe {RUTA_CORPUS}: correr el notebook 02 con esa etiqueta'
print('Corpus:', RUTA_CORPUS.name, '· textos:', RUTA_TEXTOS.name, '· modelo:', MODELO)


Corpus: corpus_v0_escala05_prior40.parquet · textos: textos_v0_escala05_prior40.jsonl · modelo: mistralai/Mistral-7B-Instruct-v0.3


## 3. Carga del corpus estructurado

Se carga el Parquet del notebook 02 junto con las semillas (para el contexto de cada episodio: origen, destino, tipo de turismo, presupuesto, activos). Se agrupa por episodio y se seleccionan los que corresponden al alcance del piloto. Los episodios que ya tengan texto en el archivo incremental se omiten.


In [42]:
import batsu_sim as bs
corpus = pd.read_parquet(RUTA_CORPUS)
semillas = bs.semillas.set_index('id')

episodios = corpus.sort_values(['id_semilla', 'corrida', 'turno']).groupby(['id_semilla', 'corrida'])
claves = list(episodios.groups.keys())
largo = episodios.size()
desenlace_ep = episodios['desenlace'].last()      # desenlace final de cada episodio

ids_sel = sorted({k[0] for k in claves})[:N_SEMILLAS] if N_SEMILLAS else sorted({k[0] for k in claves})
if N_CORRIDAS is None:
    claves = [k for k in claves if k[0] in ids_sel]
else:   # piloto: las primeras N_CORRIDAS corridas de cada semilla con al menos MIN_TURNOS_PILOTO turnos
    claves = [k for sid in ids_sel for k in [c for c in claves if c[0] == sid and largo[c] >= MIN_TURNOS_PILOTO][:N_CORRIDAS]]

def validar_turnos(turnos, n, desenlace):
    """Valida el esquema del diálogo (no solo que sea JSON): n turnos, cada uno con 'agente' y 'turista'
    de tipo texto, no vacíos (salvo el último mensaje del turista en un abandono) y sin caracteres dañados."""
    if not isinstance(turnos, list) or len(turnos) != n:
        return False, 'numero_de_turnos'
    for i, t in enumerate(turnos):
        if not isinstance(t, dict):
            return False, f'turno_{i+1}_no_es_objeto'
        for campo in ('agente', 'turista'):
            v = t.get(campo)
            if not isinstance(v, str):
                return False, f'turno_{i+1}_{campo}_tipo'
            if '\ufffd' in v:
                return False, f'turno_{i+1}_{campo}_caracteres_danados'
            vacio_permitido = campo == 'turista' and i == n - 1 and desenlace == 'abandono'
            if not v.strip() and not vacio_permitido:
                return False, f'turno_{i+1}_{campo}_vacio'
    return True, None

# Lo ya redactado solo cuenta si pasa la validación de esquema; el último registro de cada episodio prevalece
ultimo = {}
if RUTA_TEXTOS.exists():
    for linea in open(RUTA_TEXTOS, encoding='utf-8'):
        d = json.loads(linea)
        ultimo[(d['id_semilla'], d['corrida'])] = d
hechos, a_rehacer = set(), 0
for k, d in ultimo.items():
    if k in largo.index and d.get('valido') and validar_turnos(d.get('turnos'), int(largo[k]), desenlace_ep[k])[0]:
        hechos.add(k)
    else:
        a_rehacer += 1

pendientes = sorted([k for k in claves if k not in hechos], key=lambda k: largo[k])   # lotes homogéneos en longitud
print(f'Episodios en alcance: {len(claves)} · ya redactados y válidos: {len(hechos)} · '
      f'a rehacer por esquema: {a_rehacer} · pendientes: {len(pendientes)}')


Episodios en alcance: 6000 · ya redactados y válidos: 5999 · a rehacer por esquema: 1 · pendientes: 1


## 4. Carga del modelo generador

vLLM carga Mistral-7B-Instruct v0.3 en 16 bits y reserva el 90 % de la memoria de la GPU: 14.5 GB para los pesos y el resto para la caché de atención (KV cache), que es lo que le permite procesar decenas de episodios simultáneamente. `max_model_len` acota la longitud de prompt más respuesta a 4 096 tokens, suficiente para el episodio más largo (12 turnos) con margen.


In [43]:
import subprocess, requests, os

LOG = '/content/vllm_server.log'
if 'vllm_proc' not in globals() or vllm_proc.poll() is not None:
    vllm_proc = subprocess.Popen(
        ['vllm', 'serve', MODELO, '--dtype', 'half', '--max-model-len', '4096',
         '--gpu-memory-utilization', '0.85', '--enforce-eager', '--port', '8000',
         '--served-model-name', 'generador'],
        stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, env={**os.environ, 'HF_TOKEN': HF_TOKEN})

t0 = time.time()
while True:
    try:
        if requests.get('http://localhost:8000/health', timeout=2).status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass
    if vllm_proc.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError('El servidor vLLM se detuvo; ver el final del log arriba')
    time.sleep(10)
print(f'Servidor vLLM listo en {time.time()-t0:.0f} s')


Servidor vLLM listo en 0 s


## 5. Construcción del prompt a partir de la traza del episodio

El prompt sigue la estructura de rol, contexto y restricciones de salida de la Sección 4.7. Contiene:

- **Rol**: redactor de un diálogo de WhatsApp entre el turista y la MiPyme.
- **Contexto de la semilla**: segmento, origen y destino, tipo de turismo, presupuesto, activos disponibles, estado afectivo inicial con sus marcadores lingüísticos (Tabla 4.2).
- **Traza turno a turno**, ya decidida por la tabla de reacción: la acción del agente (con el activo enviado si aplica), el estado afectivo del turista antes y después, si la acción violó el Trust Gate y, en el último turno, el desenlace.
- **Restricciones**: un mensaje del agente y una respuesta del turista por turno, máximo 40 palabras cada uno, estilo WhatsApp, y salida **únicamente** en JSON con la lista de turnos.

Para el turista nacional se pide español costarricense coloquial; para el internacional, español neutro (o inglés, según el parámetro). Los estados afectivos se describen con los marcadores de la Tabla 4.2 para que el texto los refleje de forma consistente, que es lo que después el juez debe reconocer.


In [44]:
MARCADORES = {   # Tabla 4.2
    'entusiasmado': 'signos de exclamación, emojis, superlativos ("me encanta", "buenísimo")',
    'indeciso': 'preguntas comparativas, condicionales ("no sé si...", "tal vez")',
    'ansioso': 'preguntas repetidas sobre tiempo o seguridad, urgencia ("necesito confirmar ya")',
    'desconfiado': 'preguntas de verificación ("¿es oficial?", "¿tienen reseñas?"), tono cauteloso',
    'frustrado': 'quejas explícitas, lenguaje cortante, referencias a intentos previos fallidos',
}
DESCRIPCION_ACCION = {   # Tabla 5.8
    'solicitar_informacion': 'el agente pide un dato que le falta (fechas, presupuesto, número de personas)',
    'ofrecer_paquete_bajo': 'el agente ofrece un paquete económico',
    'ofrecer_paquete_medio': 'el agente ofrece un paquete de precio medio',
    'ofrecer_paquete_alto': 'el agente ofrece un paquete de precio alto',
    'enviar_activo': 'el agente envía un activo promocional',
    'escalar_humano': 'el agente transfiere la conversación a un asesor humano, que responde en ese mismo turno',
    'cerrar_conversacion': 'el agente cierra la conversación cordialmente sin nueva oferta',
}
DESENLACE_TXT = {
    'conversion': 'el turista confirma la reserva',
    'avance_parcial': 'el turista se despide interesado pero sin reservar todavía',
    'rechazo': 'el turista declina explícitamente la oferta',
    'abandono': 'el turista deja de responder (su último mensaje queda vacío o es una despedida evasiva)',
    'recomendacion_inviable': 'el turista señala que lo ofrecido es imposible para él por la distancia o el traslado, y se molesta',
    'finalizacion_limite_turnos': 'la conversación se agota sin cierre; el turista se despide sin decidir',
}

def construir_prompt(id_sem, corrida):
    s = semillas.loc[id_sem]
    tr = episodios.get_group((id_sem, corrida)).sort_values('turno')
    nacional = s['segmento'] == 'nacional'
    if nacional:
        lengua = 'español costarricense coloquial (voseo, expresiones locales como "mae", "pura vida", "diay")'
    else:
        lengua = 'español neutro, claro y sin modismos' if IDIOMA_INTERNACIONAL == 'es' else 'English (the tourist writes in English; the business answers in English)'
    activos = ', '.join(s['activos_lista'])
    lineas = []
    for r in tr.itertuples():
        det = DESCRIPCION_ACCION[r.accion_nombre]
        if r.accion_nombre == 'enviar_activo' and r.activo_enviado:
            det += f' ({r.activo_enviado})'
        if r.enmascarada:
            det = 'el agente intenta enviar un activo que no tiene; solo puede responder con texto'
        nota = ''
        if r.violacion_trust_gate and 'M2' in (r.modificadores or ''):
            nota = ' [la oferta supera el presupuesto del turista]'
        if r.violacion_trust_gate and 'M1' in (r.modificadores or ''):
            nota = ' [la oferta es geográficamente inviable para el turista]'
        fin = f' → FIN: {DESENLACE_TXT[r.desenlace]}' if r.desenlace else ''
        cambio = f'sigue {r.estado_despues}' if r.estado_antes == r.estado_despues else f'pasa de {r.estado_antes} a {r.estado_despues}'
        lineas.append(f'Turno {r.turno}: {det}{nota}. El turista {cambio}{fin}.')
    traza = '\n'.join(lineas)
    return f"""Actúe como redactor de un diálogo de WhatsApp entre un turista y una MiPyme turística de Costa Rica.
El diálogo ya está decidido turno a turno; su única tarea es redactar los mensajes con naturalidad, en {lengua}.

CONTEXTO DEL CASO
- Turista {s['segmento']}, viaja desde {s['origen']} hacia {s['destino']} ({s['tipo_turismo']}), presupuesto {s['presupuesto']}.
- Activos que la MiPyme tiene disponibles: {activos}.
- Estado afectivo inicial del turista: {s['estado_afectivo_inicial']} ({MARCADORES[s['estado_afectivo_inicial']]}).
- Cuando el estado afectivo cambia, el mensaje del turista debe reflejarlo con sus marcadores: {'; '.join(f'{k}: {v}' for k, v in MARCADORES.items())}.

TRAZA DEL EPISODIO (no la altere)
{traza}

RESTRICCIONES DE SALIDA
- Exactamente {len(tr)} turnos. En cada turno, primero escribe el agente y luego responde el turista.
- Máximo {MAX_PALABRAS_MENSAJE} palabras por mensaje, estilo WhatsApp (breve, directo, puede usar emojis con moderación).
- Cada turno debe ser distinto de los anteriores. Si la acción del agente se repite, cambie el contenido: pida otro dato, aporte un detalle nuevo del paquete o responda a lo que el turista acaba de decir. El turista nunca repite su mensaje anterior.
- No invente precios exactos ni nombres de personas; puede decir "el paquete económico", "unos 45 mil colones" es aceptable solo si es coherente con el presupuesto.
- Responda ÚNICAMENTE con JSON válido, sin texto antes ni después, con esta forma:
{{"turnos": [{{"turno": 1, "agente": "...", "turista": "..."}}, ...]}}"""

print(construir_prompt(*pendientes[0]) if pendientes else construir_prompt(*claves[0]))


Actúe como redactor de un diálogo de WhatsApp entre un turista y una MiPyme turística de Costa Rica.
El diálogo ya está decidido turno a turno; su única tarea es redactar los mensajes con naturalidad, en español costarricense coloquial (voseo, expresiones locales como "mae", "pura vida", "diay").

CONTEXTO DEL CASO
- Turista nacional, viaja desde Grecia hacia Ojo de Agua (aventura), presupuesto medio.
- Activos que la MiPyme tiene disponibles: texto, imagen, video.
- Estado afectivo inicial del turista: desconfiado (preguntas de verificación ("¿es oficial?", "¿tienen reseñas?"), tono cauteloso).
- Cuando el estado afectivo cambia, el mensaje del turista debe reflejarlo con sus marcadores: entusiasmado: signos de exclamación, emojis, superlativos ("me encanta", "buenísimo"); indeciso: preguntas comparativas, condicionales ("no sé si...", "tal vez"); ansioso: preguntas repetidas sobre tiempo o seguridad, urgencia ("necesito confirmar ya"); desconfiado: preguntas de verificación ("¿es ofi

## 6. Redacción por lotes con guardado incremental

Cada lote de episodios se convierte en conversaciones (un mensaje de usuario con el prompt) y se entrega completo a `llm.chat`, que aplica la plantilla de chat de Mistral y genera todas las respuestas con continuous batching. Se muestrea con temperatura 0.7. Cada respuesta se valida en dos niveles: que sea JSON y que cumpla el esquema (validar_turnos): el número de turnos esperado, mensajes de tipo texto, sin campos vacíos (salvo el último mensaje del turista en un abandono) y sin caracteres dañados. Si falla, el episodio se reintenta una vez; si vuelve a fallar queda marcado con valido: false y el motivo. Los episodios ya redactados que no pasan el esquema se vuelven a redactar en la siguiente corrida.

Cada lote se escribe de inmediato en `textos_<etiqueta>.jsonl` y se imprime el tiempo acumulado con la proyección para el corpus completo.


In [45]:
!pip install -q openai 2>&1 | tail -1
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor

cliente = OpenAI(base_url='http://localhost:8000/v1', api_key='no-key')

def extraer_json(texto):
    m = re.search(r'\{.*\}', texto or '', re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

def esquema_dialogo(n):
    """Esquema JSON con exactamente n turnos: vLLM restringe la salida a esta forma (decodificación guiada)."""
    turno = {'type': 'object',
             'properties': {'turno': {'type': 'integer'},
                            'agente': {'type': 'string', 'minLength': 1},
                            'turista': {'type': 'string', 'minLength': 1}},
             'required': ['turno', 'agente', 'turista'], 'additionalProperties': False}
    return {'type': 'object',
            'properties': {'turnos': {'type': 'array', 'items': turno, 'minItems': n, 'maxItems': n}},
            'required': ['turnos'], 'additionalProperties': False}

def _uno(prompt, n):
    r = cliente.chat.completions.create(
        model='generador', messages=[{'role': 'user', 'content': prompt}],
        temperature=TEMPERATURA, top_p=0.9, max_tokens=min(2400, 160 * n + 400),
        response_format={'type': 'json_schema', 'json_schema': {'name': 'dialogo', 'schema': esquema_dialogo(n)}})
    return r.choices[0].message.content

def generar_lote(prompts, ns):
    with ThreadPoolExecutor(max_workers=min(LOTE, len(prompts))) as ex:
        return list(ex.map(lambda pn: _uno(*pn), zip(prompts, ns)))

def redactar(claves_lote, intento=1):
    prompts = [construir_prompt(*k) for k in claves_lote]
    n_turnos = [int(largo[k]) for k in claves_lote]
    respuestas = generar_lote(prompts, n_turnos)
    resultados, fallidos = [], []
    for k, n, resp in zip(claves_lote, n_turnos, respuestas):
        d = extraer_json(resp)
        turnos = d.get('turnos') if isinstance(d, dict) else None
        ok, motivo = validar_turnos(turnos, n, desenlace_ep[k]) if d is not None else (False, 'json_invalido')
        if ok:
            resultados.append({'id_semilla': k[0], 'corrida': k[1], 'valido': True, 'turnos': turnos,
                               'modelo': MODELO, 'intento': intento})
        elif intento == 1:
            fallidos.append(k)
        else:
            resultados.append({'id_semilla': k[0], 'corrida': k[1], 'valido': False, 'turnos': None, 'motivo': motivo,
                               'crudo': (resp or '')[:2000], 'modelo': MODELO, 'intento': intento})
    if fallidos:
        resultados += redactar(fallidos, intento=2)
    return resultados

total = len(pendientes); hechos_ahora = 0; t0 = time.time()
n_corpus = corpus.groupby(['id_semilla', 'corrida']).ngroups
print(f'Pendientes: {total}')
with open(RUTA_TEXTOS, 'a', encoding='utf-8') as f:
    for i in range(0, total, LOTE):
        lote = pendientes[i:i + LOTE]
        for r in redactar(lote):
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
        f.flush()
        hechos_ahora += len(lote)
        seg = time.time() - t0
        por_ep = seg / hechos_ahora
        print(f'{hechos_ahora}/{total} episodios · {seg/60:.1f} min · {por_ep:.2f} s/episodio · '
              f'restante estimado: {por_ep*(total-hechos_ahora)/3600:.2f} h')
print('Redacción terminada' if total else 'No había episodios pendientes')

Pendientes: 1
1/1 episodios · 0.1 min · 8.26 s/episodio · restante estimado: 0.00 h
Redacción terminada


## 7. Validación, muestra y manifiesto

Se relee el archivo incremental, se cuentan los episodios válidos e inválidos, se verifica la longitud de los mensajes contra el límite de 40 palabras y se muestra un episodio completo para revisar la calidad: que el estado afectivo se note en el texto, que la acción del agente coincida con la traza y que el cierre corresponda al desenlace.

Finalmente se actualiza el manifiesto del corpus con el modelo, la versión, los parámetros de generación y los conteos.


In [46]:
import hashlib, inspect
from collections import Counter

# Último registro de cada episodio (los rehechos reemplazan a los anteriores)
ultimo = {}
for l in open(RUTA_TEXTOS, encoding='utf-8'):
    d = json.loads(l)
    ultimo[(d['id_semilla'], d['corrida'])] = d

validos, motivos = [], Counter()
for k, r in ultimo.items():
    if r.get('valido'):
        ok, motivo = validar_turnos(r.get('turnos'), int(largo[k]), desenlace_ep[k])
    else:
        ok, motivo = False, r.get('motivo') or 'json_invalido'
    if ok:
        validos.append(r)
    else:
        motivos[re.sub(r'turno_\d+_', '', motivo or 'desconocido')] += 1

print(f'Episodios redactados: {len(ultimo)} · válidos (JSON + esquema): {len(validos)} · inválidos: {len(ultimo) - len(validos)}')
if motivos:
    print('Motivos de invalidez:', dict(motivos))

palabras = [len(t[c].split()) for r in validos for t in r['turnos'] for c in ('agente', 'turista') if t.get(c)]
if palabras:
    print(f'Palabras por mensaje: media {np.mean(palabras):.1f} · máx {max(palabras)} · > {MAX_PALABRAS_MENSAJE}: '
          f'{np.mean(np.array(palabras) > MAX_PALABRAS_MENSAJE):.1%}')

# Muestra: último episodio válido, con su traza al lado (revisión manual de coherencia acción ↔ texto)
if validos:
    r = validos[-1]; k = (r['id_semilla'], r['corrida']); tr = episodios.get_group(k).sort_values('turno')
    s = semillas.loc[k[0]]
    print(f'\n=== {k[0]} corrida {k[1]} · {s["segmento"]} · {s["tipo_turismo"]} · {s["origen"]} → {s["destino"]} · esperado: {s["resultado_esperado"]} ===')
    for t, row in zip(r['turnos'], tr.itertuples()):
        print(f'[{row.turno}] ({row.accion_nombre}; {row.estado_antes}→{row.estado_despues}{" · " + row.desenlace if row.desenlace else ""})')
        print(f'    MiPyme:  {t.get("agente")}')
        print(f'    Turista: {t.get("turista")}')

# Huella del prompt: código de construir_prompt y los textos que lo componen
try:
    fuente_prompt = inspect.getsource(construir_prompt)
except (OSError, TypeError):
    fuente_prompt = ''
fuente_prompt += json.dumps([MARCADORES, DESCRIPCION_ACCION, DESENLACE_TXT], ensure_ascii=False, sort_keys=True)
hash_prompt = hashlib.sha256(fuente_prompt.encode('utf-8')).hexdigest()[:16]

# Manifiesto
ruta_manif = CORPUS_DIR / f'manifiesto_{ETIQUETA_CORPUS}.json'
manif = json.load(open(ruta_manif, encoding='utf-8'))
manif['generador'] = {'modelo': MODELO, 'precision': 'fp16', 'motor': 'vllm ' + __import__('vllm').__version__,
                      'temperatura': TEMPERATURA, 'top_p': 0.9,
                      'idioma_internacional': IDIOMA_INTERNACIONAL, 'max_palabras_mensaje': MAX_PALABRAS_MENSAJE,
                      'prompt_sha256': hash_prompt, 'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
                      'episodios_redactados': len(ultimo), 'episodios_validos': len(validos),
                      'validacion_esquema': {'invalidos': len(ultimo) - len(validos), 'motivos': dict(motivos)},
                      'completo': len(validos) == corpus.groupby(['id_semilla', 'corrida']).ngroups}
json.dump(manif, open(ruta_manif, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('\nManifiesto actualizado:', ruta_manif.name, '· huella del prompt:', hash_prompt)

Episodios redactados: 6000 · válidos (JSON + esquema): 6000 · inválidos: 0
Palabras por mensaje: media 16.1 · máx 75 · > 40: 0.6%

=== CR-TUR-053 corrida 22 · nacional · rural · Puntarenas → Bijagua · esperado: abandono ===
[1] (solicitar_informacion; ansioso→indeciso)
    MiPyme:  Hola 🌱! ¡Qué pena que no tengo tu presupuesto y número de personas! 😕 Si me lo das, te podré ayudar con un paquete a Bijagua. 😊
    Turista: Soy de Puntarenas y voy solo. Quiero ir a Bijagua, pero no sé cuanto durará mi estadía. ¿Cuánto cuesta un paquete económico? 😟
[2] (solicitar_informacion; indeciso→indeciso)
    MiPyme:  ¡Bienvenido a Bijagua! 🌳 El precio de un paquete económico depende de la duración, pero te lo dare a ver. 😊 ¿Por cuanto tiempo quieres ir? 😃
    Turista: No sé por cuánto tiempo quiero quedarme, pero necesito confirmar ya. 😟
[3] (enviar_activo; indeciso→entusiasmado)
    MiPyme:  ¡Entendido! 🌺 Aquí te dejo un par de fotos de Bijagua para que veas qué esperar. 😊 https://i.imgur.com/12345

In [47]:
# Trazabilidad: los episodios rehechos después del primer pase usaron decodificación guiada por esquema JSON
manif = json.load(open(ruta_manif, encoding='utf-8'))
manif['generador']['decodificacion_guiada'] = {
    'metodo': 'vLLM response_format json_schema (exactamente N turnos)',
    'episodios': 644,
    'sin_guiada': 5356,
    'motivo': 'el modelo ignoraba el numero de turnos en episodios cortos; el prompt no cambia (misma huella)',
}
json.dump(manif, open(ruta_manif, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(json.dumps(manif['generador'], ensure_ascii=False, indent=2))

{
  "modelo": "mistralai/Mistral-7B-Instruct-v0.3",
  "precision": "fp16",
  "motor": "vllm 0.31.0",
  "temperatura": 0.7,
  "top_p": 0.9,
  "idioma_internacional": "es",
  "max_palabras_mensaje": 40,
  "prompt_sha256": "6cb6529860c93993",
  "fecha": "2026-10-14 07:55:32",
  "episodios_redactados": 6000,
  "episodios_validos": 6000,
  "validacion_esquema": {
    "invalidos": 0,
    "motivos": {}
  },
  "completo": true,
  "decodificacion_guiada": {
    "metodo": "vLLM response_format json_schema (exactamente N turnos)",
    "episodios": 644,
    "sin_guiada": 5356,
    "motivo": "el modelo ignoraba el numero de turnos en episodios cortos; el prompt no cambia (misma huella)"
  }
}
